In [ ]:
import pandas as pd
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from collections import Counter
import re
import random
import matplotlib
from PIL import Image
import numpy as np

# 서버 환경이나 터미널 실행 시 충돌 방지
matplotlib.use('Agg')

# ===============================
# 1. 데이터 로드 (파일명 확인 필수!)
# ===============================
df = pd.read_csv('./data/bx_crawling_results_판단.csv') # 저장하신 파일명으로 수정
full_text = " ".join(df['title'].astype(str)) + " " + " ".join(df['snippet'].astype(str))

# ===============================
# 2. 텍스트 정제 (2글자 이상 한글만)
# ===============================
words = re.findall(r'[가-힣]{2,}', full_text)

# ===============================
# 3. 인지적 부양 관련 필터링 키워드 (제외 및 포함)
# ===============================
# 분석과 무관한 광고성/일반 단어 제거
exclude_patterns = [
    '네이버', '블로그', '카페', '바로가기', '저장', '이미지', '뉴스', '연말정산',
    '인적공제', '부양가족', '신청방법', '관련', '내용', '경우', '대한', '통해'
]

# 인지적 부양의 핵심 테마 (이 단어들이 포함된 것만 추출)
theme_keywords = [
    '스마트폰', '가르치', '반복', '대신', '예매', '결제', '인증', '확인', 
    '스트레스', '답답', '화남', '결국', '방문', '원격', '전화', '엄마', '아빠',
    '부모님', '가전', '사용법', '질문', '또물어', '노동', '불안', '강박'
]

# ===============================
# 4. 빈도 계산 및 필터링
# ===============================
word_counts = Counter(words)
final_counts = {}

for word, count in word_counts.items():
    if any(ex in word for ex in exclude_patterns):
        continue
    if any(theme in word for theme in theme_keywords):
        final_counts[word] = count

# ===============================
# 5. 마스크 설정 (이미지 모양대로 배치)
# ===============================
# 마스크 이미지가 있는 경로를 지정하세요 (없으면 마스크 부분 주석 처리 가능)
try:
    mask_img = Image.open('./data/iphone.jpg').convert('RGB') # 원하는 모양의 이미지
    mask_np = np.array(mask_img)
    mask = np.where((mask_np[:, :, 0] > 240), 255, 0)
except:
    print("마스크 이미지를 찾을 수 없어 기본 사각형으로 진행합니다.")
    mask = None

# ===============================
# 6. 컬러 테마 (신뢰감 있는 다크블루 + 경고성 레드/그레이 조합)
# ===============================
custom_colors = [
    "#1A237E", # 다크 블루 (기업용 신뢰감)
    "#C62828", # 레드 (자녀의 고충/경고)
    "#455A64", # 그레이 (인지적 부담감)
    "#E91E63", # 핑크계열 (가족 관계)
    "#1565C0"
]

def custom_color_func(*args, **kwargs):
    return random.choice(custom_colors)

# ===============================
# 7. 워드클라우드 생성
# ===============================
wordcloud = WordCloud(
    font_path='C:/Windows/Fonts/malgun.ttf', # 윈도우 맑은고딕
    background_color='white',
    width=1200,
    height=1200,
    mask=mask,
    max_words=300,
    max_font_size=180,
    min_font_size=10,
    relative_scaling=0.2, # 단어 빈도에 따른 크기 차이 조절
    prefer_horizontal=0.8, # 가로쓰기 비중
    color_func=custom_color_func,
    margin=2,
    contour_width=10,  # 2 → 0 (윤곽선 제거)
    contour_color='black'
).generate_from_frequencies(final_counts)

# ===============================
# 8. 저장 및 시각화
# ===============================
plt.figure(figsize=(10, 10))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.tight_layout(pad=0)
plt.savefig('cognitive_labor_cloud.png', dpi=300, bbox_inches='tight')
print("✨ 워드클라우드 저장이 완료되었습니다: cognitive_labor_cloud.png")